# Composition-Based Data Preparation for Ferroelectric Classification

## Objective

This notebook prepares the Matminer `dielectric_constant` dataset for
composition-based machine-learning analysis.

Chemical formulas are converted into elemental-fraction descriptors and
combined with the `pot_ferroelectric` classification target.

The downstream analysis examines the predictive information available from
chemical composition alone. Structural information is deliberately excluded
from the present feature representation.

## 1. Setup

Only the packages required for data preparation are imported here. Package versions are printed so that changes in the underlying Matminer dataset or featurization API can be documented.

In [16]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matminer

from matminer.datasets import load_dataset
from matminer.featurizers.base import MultipleFeaturizer
from matminer.featurizers.composition import ElementFraction
from matminer.featurizers.conversions import StrToComposition

RANDOM_STATE = 42

DATA_DIR = Path("../data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Python: {sys.version.split()[0]}")
print(f"NumPy: {np.__version__}")
print(f"pandas: {pd.__version__}")
print(f"Matminer: {matminer.__version__}")

Python: 3.10.21
NumPy: 2.2.6
pandas: 2.3.3
Matminer: 0.9.3


## 2. Load and inspect dataset

The public `dielectric_constant` dataset is used as the starting point. The dataset contains composition and property information for inorganic materials, including a boolean `pot_ferroelectric` label and band-gap values.

The exact number of rows or available fields may vary if the distributed dataset changes between Matminer versions. For that reason, the dataset shape is recorded programmatically rather than hard-coded.

In [17]:
df = load_dataset("dielectric_constant")

print(f"Dataset shape: {df.shape}")
display(df.head())

Dataset shape: (1056, 16)


,material_id,formula,nsites,space_group,volume,structure,band_gap,e_electronic,e_total,n,poly_electronic,poly_total,pot_ferroelectric,cif,meta,poscar
0,mp-441,Rb2Te,3,225,159.501208,"[[1.75725875 1.2425695 3.04366125] Rb, [5.271...",1.88,"[[3.44115795, -3.097e-05, -6.276e-05], [-2.837...","[[6.23414745, -0.00035252, -9.796e-05], [-0.00...",1.86,3.44,6.23,False,#\#CIF1.1\n###################################...,{u'incar': u'NELM = 100\nIBRION = 8\nLWAVE = F...,Rb2 Te1\n1.0\n5.271776 0.000000 3.043661\n1.75...
1,mp-22881,CdCl2,3,166,84.298097,"[[0. 0. 0.] Cd, [ 4.27210959 2.64061969 13.13...",3.52,"[[3.34688382, -0.04498543, -0.22379197], [-0.0...","[[7.97018673, -0.29423886, -1.463590159999999]...",1.78,3.16,6.73,False,#\#CIF1.1\n###################################...,{u'incar': u'NELM = 100\nIBRION = 8\nLWAVE = F...,Cd1 Cl2\n1.0\n3.850977 0.072671 5.494462\n1.78...
2,mp-28013,MnI2,3,164,108.335875,"[[0. 0. 0.] Mn, [-2.07904300e-06 2.40067320e+...",1.17,"[[5.5430849, -5.28e-06, -2.5030000000000003e-0...","[[13.80606079, 0.0006911900000000001, 9.655e-0...",2.23,4.97,10.64,False,#\#CIF1.1\n###################################...,{u'incar': u'NELM = 100\nIBRION = 8\nLWAVE = F...,Mn1 I2\n1.0\n4.158086 0.000000 0.000000\n-2.07...
3,mp-567290,LaN,4,186,88.162562,[[-1.73309900e-06 2.38611186e+00 5.95256328e...,1.12,"[[7.09316738, 7.99e-06, -0.0003864700000000000...","[[16.79535386, 8.199999999999997e-07, -0.00948...",2.65,7.04,17.99,False,#\#CIF1.1\n###################################...,{u'incar': u'NELM = 100\nIBRION = 8\nLWAVE = F...,La2 N2\n1.0\n4.132865 0.000000 0.000000\n-2.06...
4,mp-560902,MnF2,6,136,82.826401,"[[1.677294 2.484476 2.484476] Mn, [0. 0. 0.] M...",2.87,"[[2.4239622, 7.452000000000001e-05, 6.06100000...","[[6.44055613, 0.0020446600000000002, 0.0013203...",1.53,2.35,7.12,False,#\#CIF1.1\n###################################...,{u'incar': u'NELM = 100\nIBRION = 8\nLDAUTYPE ...,Mn2 F4\n1.0\n3.354588 0.000000 0.000000\n0.000...


In [18]:
print("Available columns:")
for col in df.columns:
    print(f"  - {col}")

Available columns:
  - material_id
  - formula
  - nsites
  - space_group
  - volume
  - structure
  - band_gap
  - e_electronic
  - e_total
  - n
  - poly_electronic
  - poly_total
  - pot_ferroelectric
  - cif
  - meta
  - poscar


## 3. Target definition

The classification target is `pot_ferroelectric`. The `band_gap` field is retained for possible later regression studies, but it is **not** used as an input feature for the ferroelectric classification task.

The machine-learning descriptors in this case study are derived from chemical composition only.

In [19]:
required_columns = ["formula", "pot_ferroelectric", "band_gap"]
missing_columns = [col for col in required_columns if col not in df.columns]

if missing_columns:
    raise KeyError(f"Required columns missing from dataset: {missing_columns}")

preview_columns = [col for col in ["material_id", "formula", "band_gap", "pot_ferroelectric"] if col in df.columns]
display(df[preview_columns].head())

,material_id,formula,band_gap,pot_ferroelectric
0,mp-441,Rb2Te,1.88,False
1,mp-22881,CdCl2,3.52,False
2,mp-28013,MnI2,1.17,False
3,mp-567290,LaN,1.12,False
4,mp-560902,MnF2,2.87,False


## 4. Composition featurization

The original dataset stores `pot_ferroelectric` as a boolean variable. It is converted to an integer target (`0` or `1`) for downstream machine learning.

Class balance is inspected before model development because strong imbalance can make metrics such as accuracy misleading.

In [20]:
target = df["pot_ferroelectric"].astype(int).rename("target_ferroelectric")

class_counts = target.value_counts().sort_index()
class_fractions = target.value_counts(normalize=True).sort_index()

class_summary = pd.DataFrame({
    "count": class_counts,
    "fraction": class_fractions,
})

class_summary.index = [
    "not_potentially_ferroelectric" if idx == 0 else "potentially_ferroelectric"
    for idx in class_summary.index
]

display(class_summary)

,count,fraction
not_potentially_ferroelectric,347,0.328598
potentially_ferroelectric,709,0.671402


## 5. Data-quality checks

Before featurization, the notebook checks for missing formulas, duplicated formulas, and missing target values.

Duplicate formulas are reported rather than automatically removed because identical compositions can correspond to different crystal structures or entries. Any deduplication decision should therefore be made with physical context rather than purely syntactic criteria.

In [21]:
print(f"Missing formulas: {df['formula'].isna().sum()}")
print(f"Missing targets: {target.isna().sum()}")
print(f"Duplicated formulas: {df['formula'].duplicated().sum()}")

Missing formulas: 0
Missing targets: 0
Duplicated formulas: 92


## 6. Export

Matminer's `StrToComposition` featurizer converts formula strings into composition objects. These objects provide a structured representation of elemental stoichiometry and are used as the input for composition featurization.

In [22]:
id_columns = [col for col in ["material_id", "formula"] if col in df.columns]
formula_df = df[id_columns].copy()

converter = StrToComposition(target_col_id="composition")
composition_df = converter.featurize_dataframe(
    formula_df,
    col_id="formula",
    pbar=False,
)

display(composition_df.head())

,material_id,formula,composition
0,mp-441,Rb2Te,"(Rb, Te)"
1,mp-22881,CdCl2,"(Cd, Cl)"
2,mp-28013,MnI2,"(Mn, I)"
3,mp-567290,LaN,"(La, N)"
4,mp-560902,MnF2,"(Mn, F)"


## 7. Generate elemental-fraction descriptors

Each material is represented by the fractional abundance of each chemical element using Matminer's `ElementFraction` featurizer.

This produces an interpretable composition vector: each feature corresponds to an element, and the feature value gives that element's fraction in the composition.

In [23]:
featurizer = MultipleFeaturizer([
    ElementFraction(),
])

feature_values = featurizer.featurize_many(
    composition_df["composition"],
    pbar=False,
)

X = pd.DataFrame(
    feature_values,
    columns=featurizer.feature_labels(),
    index=df.index,
)

print(f"Raw composition feature matrix shape: {X.shape}")
display(X.head())

Raw composition feature matrix shape: (1056, 118)


,H,He,Li,Be,B,C,N,O,F,Ne,...,Mt,Ds,Rg,Cn,Nh,Fl,Mc,Lv,Ts,Og
0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0,...,0,0,0,0,0,0,0,0,0,0
1,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0,...,0,0,0,0,0,0,0,0,0,0
2,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0,...,0,0,0,0,0,0,0,0,0,0
3,0.0,0,0.0,0.0,0.0,0.0,0.5,0.0,0.000000,0,...,0,0,0,0,0,0,0,0,0,0
4,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.666667,0,...,0,0,0,0,0,0,0,0,0,0


## 8. Inspect descriptor quality

The feature matrix is checked for:

- missing numerical values,
- duplicate feature rows,
- infinite values,
- zero-variance (constant) elemental features.

Zero-variance features are identified here but retained in the exported dataset. Feature filtering is performed within the supervised-learning pipeline using `VarianceThreshold`.

In [24]:
missing_feature_values = int(X.isna().sum().sum())
duplicate_feature_rows = int(X.duplicated().sum())
infinite_feature_values = int(np.isinf(X.to_numpy(dtype=float)).sum())

constant_features = X.columns[X.nunique(dropna=False) <= 1].tolist()

print(f"Missing feature values: {missing_feature_values}")
print(f"Infinite feature values: {infinite_feature_values}")
print(f"Duplicate feature rows: {duplicate_feature_rows}")
print(f"Constant features: {len(constant_features)}")

if constant_features:
    print("\nConstant feature names:")
    print(", ".join(constant_features))

Missing feature values: 0
Infinite feature values: 0
Duplicate feature rows: 92
Constant features: 55

Constant feature names:
He, Ne, Ar, Kr, Tc, Xe, Ce, Pr, Nd, Pm, Sm, Eu, Gd, Tb, Dy, Ho, Er, Tm, Yb, Lu, Po, At, Rn, Fr, Ra, Ac, Th, Pa, U, Np, Pu, Am, Cm, Bk, Cf, Es, Fm, Md, No, Lr, Rf, Db, Sg, Bh, Hs, Mt, Ds, Rg, Cn, Nh, Fl, Mc, Lv, Ts, Og


In [25]:
print(f"Raw elemental feature count: {X.shape[1]}")
print(f"Globally constant features identified: {len(constant_features)}")
print("Constant-feature removal will be performed later inside the ML pipeline.")

Raw elemental feature count: 118
Globally constant features identified: 55
Constant-feature removal will be performed later inside the ML pipeline.


## 9. Verify elemental-fraction representation

For a correctly constructed elemental-fraction vector, the elemental fractions of each material should sum to approximately one.


In [26]:
row_sums = X.sum(axis=1)

print(row_sums.describe())
print(f"Maximum absolute deviation from 1: {(row_sums - 1.0).abs().max():.3e}")

count    1.056000e+03
mean     1.000000e+00
std      2.940356e-17
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
dtype: float64
Maximum absolute deviation from 1: 1.110e-16


## 10. Assemble the processed dataset

The exported table contains:

- material identifier, when available,
- chemical formula,
- the complete elemental-fraction descriptor matrix,
- the binary ferroelectric target,
- band gap retained as a separate property for possible regression analysis.

All composition descriptors are exported, including descriptors that are constant across the full dataset. Constant-feature removal is deferred to the downstream machine-learning pipeline so that it can be performed within cross-validation.

In [27]:
metadata_columns = [col for col in ["material_id", "formula"] if col in df.columns]

processed_df = pd.concat(
    [
        df[metadata_columns].reset_index(drop=True),
        X.reset_index(drop=True),
        target.reset_index(drop=True),
        df[["band_gap"]].reset_index(drop=True),
    ],
    axis=1,
)

print(f"Processed dataset shape: {processed_df.shape}")
display(processed_df.head())

Processed dataset shape: (1056, 122)


,material_id,formula,H,He,Li,Be,B,C,N,O,...,Rg,Cn,Nh,Fl,Mc,Lv,Ts,Og,target_ferroelectric,band_gap
0,mp-441,Rb2Te,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,1.88
1,mp-22881,CdCl2,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,3.52
2,mp-28013,MnI2,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,1.17
3,mp-567290,LaN,0.0,0,0.0,0.0,0.0,0.0,0.5,0.0,...,0,0,0,0,0,0,0,0,0,1.12
4,mp-560902,MnF2,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,2.87


## 11. Final validation before export

A few assertions are included to make failures explicit rather than silently writing a malformed dataset.

In [28]:
assert len(processed_df) == len(df), "Row count changed during preprocessing."
assert processed_df["target_ferroelectric"].isna().sum() == 0, "Target contains missing values."
assert not X.isna().any().any(), "Feature matrix contains missing values."
assert np.isfinite(X.to_numpy(dtype=float)).all(), "Feature matrix contains non-finite values."

print("Validation checks passed.")

Validation checks passed.


## 12. Export processed data


In [29]:
output_path = DATA_DIR / "dielectric_composition_features.csv"
processed_df.to_csv(output_path, index=False)

print(f"Saved processed dataset to: {output_path.resolve()}")

Saved processed dataset to: /smb/twallis/m/data/dielectric_composition_features.csv


## 13. Reproducibility summary

Run this cell after preprocessing to record the dimensions of the dataset used by subsequent notebooks. This is particularly useful because dataset content can change between package versions.

In [30]:
summary = pd.Series({
    "n_materials": len(processed_df),
    "n_raw_elemental_features": X.shape[1],
    "n_constant_features_identified": len(constant_features),
    "n_exported_elemental_features": X.shape[1],
    "positive_class_fraction": target.mean(),
})

display(summary.to_frame("value"))

,value
n_materials,1056.000000
n_raw_elemental_features,118.000000
n_constant_features_identified,55.000000
n_exported_elemental_features,118.000000
positive_class_fraction,0.671402


## Summary and limitations

This notebook converts chemical formulas from the Matminer dielectric dataset into elemental-fraction descriptors suitable for machine-learning analysis.

The processed dataset contains:

- material identifiers and formulas,
- the complete composition-based numerical descriptor matrix,
- a binary potential-ferroelectric classification target,
- band-gap values retained for possible regression studies.

Constant elemental descriptors are identified and reported here, but they are intentionally retained in the exported dataset. Their removal is delegated to the downstream supervised-learning pipeline, where `VarianceThreshold` can be applied within cross-validation.

The next notebook uses these descriptors for supervised classification.

### Scientific limitations

Elemental fractions encode chemical composition but not crystal structure, symmetry, atomic arrangement, defects, temperature, processing conditions, or measurement uncertainty. Consequently, downstream models should be interpreted as **composition-based predictors**, not complete physical models of ferroelectric behaviour.

Duplicate compositions also require physical interpretation: two database entries with the same nominal formula may correspond to different structures or material states. They should not automatically be treated as redundant observations.

Finally, the exact dataset size and descriptor count are recorded from the installed Matminer version rather than assumed from earlier reports. This keeps the workflow reproducible if the distributed dataset changes over time.